In [ ]:
import numpy as np
import pandas as pd
import requests
import duckdb
import os
import s3fs
np.random.seed(123)

# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})
BUCKET = "mathildedonnenfeld/diffusion/bistrots_et_bulletins"

In [ ]:
FILE_KEY_S3 = "bistrots_sirene.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
    bistrots_sirene = pd.read_csv(file_in, sep=",")

bistrots_sirene["date_dernier_traitement"] = pd.to_datetime(bistrots_sirene.dateDernierTraitementEtablissement).dt.normalize()
bistrots_sirene["date_creation"] = pd.to_datetime(bistrots_sirene.dateCreationEtablissement).dt.normalize()

# Décomptes
cond_date = "2026-03-01"
print(bistrots_sirene["siret"].nunique())
print(bistrots_sirene[(bistrots_sirene['dateCreationEtablissement'] <= cond_date)]["siret"].nunique())

bistrots_sirene.head()

FILE_KEY_S3 = "filosofi.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
    filosofi = pd.read_csv(file_in, sep=",")

filosofi.head()

In [ ]:
bistrots_par_ville = (
    bistrots_sirene
    .assign(
        age=(
            pd.Timestamp("2026-03-01")
            - pd.to_datetime(
                bistrots_sirene["dateCreationEtablissement"],
                errors="coerce"
            )
        ).dt.days / 365.25
    )
    .groupby(["codeCommuneEtablissement"])
    .agg(
        nb_etablissements=("siret", "nunique"),
        age_moyen_bistrots=("age", "mean")
    )
    .reset_index()
    .sort_values("nb_etablissements", ascending=False)
)


In [ ]:
table_analyse = filosofi.merge(bistrots_par_ville, left_on="CODGEO", right_on="codeCommuneEtablissement", how="left")print(filosofi.shape[0])
print(bistrots_par_ville["codeCommuneEtablissement"].nunique())
print(bistrots_par_ville.shape[0])
print(table_analyse.shape[0])

In [ ]:
import seaborn as sns

sns.scatterplot(data=table_analyse, x="nb_etablissements", y="POPULATION", hue="NIVVIE_MEDIAN")

In [ ]:
sns.scatterplot(data=table_analyse, x="nb_etablissements", y="POPULATION")